In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, classification_report, ConfusionMatrixDisplay,
                             RocCurveDisplay, confusion_matrix)
from preprocessing import load_data, split_data, make_preprocessor, RANDOM_STATE

FIG_DIR = Path("results/david_logistic_regression")
FIG_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
X, y = load_data()
X_train, X_test, y_train, y_test = split_data(X, y)
print("Training records:", len(X_train), "Test records:", len(X_test))
print("Training positive rate:", round(y_train.mean(), 3))
print("Test positive rate:", round(y_test.mean(), 3))

Training records: 614 Test records: 154
Training positive rate: 0.349
Test positive rate: 0.351


In [ ]:
grid = GridSearchCV(
    make_lr_pipeline(),
    param_grid={"lr__C": [0.01, 0.1, 1, 10, 100],
                "lr__class_weight": [None, "balanced"]},
    scoring=scoring, refit="roc_auc", cv=cv, n_jobs=-1,
    return_train_score=True
)
grid.fit(X_train, y_train)
print("Best hyperparameters:", grid.best_params_)
print("Best CV ROC-AUC:", round(grid.best_score_, 3))

best_idx = grid.best_index_
comparison = pd.DataFrame({
    "Baseline CV": baseline_summary,
    "Tuned CV": pd.Series({m: grid.cv_results_[f"mean_test_{m}"][best_idx]
                            for m in scoring})
}).round(3)
display(comparison)